In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict
import json
import os
import pickle
import sys
sys.path.append("../../../src/experiment_util") 
import experiment_utils
sys.path.append("../../../irl") 
import data_corruption_utils

In [ ]:
with open("../../../dir_paths.json") as f:
    config = json.load(f)
    dataframes_path = config["dataframes_path"]
    project_output_path = config["project_output_path"]

first_n_df = pd.read_pickle(dataframes_path + "first_n_df.pkl")

In [ ]:
misclassified_users_rf_all_subs_dict = {}
misclassified_users_xgb_all_subs_dict = {}

empirical_pol_results = []

for n in first_n_df.n.unique():
    running_confusion_matrix_rf = np.zeros((2, 2), dtype=int)
    running_confusion_matrix_xgb = np.zeros((2, 2), dtype=int)

    f1_list_rf = []
    f1_list_xgb = []

    misclassified_users_rf_all_subs = defaultdict(int)
    misclassified_users_xgb_all_subs = defaultdict(int)
    for run in first_n_df.run.unique():
        
        print(n,"run:",run)
        seed = 100 + run


        df_troll_sampled = first_n_df[(first_n_df.run == run) & (first_n_df.n == n) & (first_n_df.russian == 1)].copy()
        df_organic_sampled = first_n_df[(first_n_df.run == run) & (first_n_df.n == n) & (first_n_df.russian == 0)].copy()

        df_troll_sampled["feature_col"] = df_troll_sampled['traj_counts_n'].apply(data_corruption_utils.normalize_replace_zeros)
        df_organic_sampled["feature_col"] = df_organic_sampled['traj_counts_n'].apply(data_corruption_utils.normalize_replace_zeros)

        confusion_matrix_rf, confusion_matrix_xgb, overall_rf_f1, overall_xgb_f1, misclassified_users_rf, misclassified_users_xgb = experiment_utils.run_experiment(df_troll_sampled,df_organic_sampled,seed = seed)

        f1_list_rf.append(overall_rf_f1)
        f1_list_xgb.append(overall_xgb_f1)
        running_confusion_matrix_rf += confusion_matrix_rf
        running_confusion_matrix_xgb += confusion_matrix_xgb
        for user in misclassified_users_rf:
            misclassified_users_rf_all_subs[user] += misclassified_users_rf[user]
        for user in misclassified_users_xgb:
            misclassified_users_xgb_all_subs[user] += misclassified_users_xgb[user]

    print("n", n)
    print("Final Confusion Matrix (Random Forest):\n", running_confusion_matrix_rf)    
    f1 = np.mean(f1_list_rf)
    print(f"Mean F1 Score (Random Forest): {f1:.2f}\n")

    print("Final Confusion Matrix (XGBoost):\n", running_confusion_matrix_xgb)        
    f1 = np.mean(f1_list_xgb)
    print(f"Mean F1 Score (XGBoost): {f1:.2f}\n")


    results = {
        "n":n,
        "f1_list_rf":f1_list_rf,
        "f1_list_xgb":f1_list_xgb,
        "running_confusion_matrix_rf":running_confusion_matrix_rf,
        "running_confusion_matrix_xgb":running_confusion_matrix_xgb,
        "misclassified_users_rf_all_subs":misclassified_users_rf_all_subs,
        "misclassified_users_xgb_all_subs":misclassified_users_xgb_all_subs
    }

    misclassified_users_rf_all_subs_dict[n] = misclassified_users_rf_all_subs
    misclassified_users_xgb_all_subs_dict[n] = misclassified_users_xgb_all_subs

    empirical_pol_results.append(results)

output_dir = project_output_path + "/early_detection"
os.makedirs(output_dir, exist_ok=True)
with open(output_dir +'/empirical_pol_results.pkl', 'wb') as f:
    pickle.dump(empirical_pol_results, f)
